# Phase 1 - Load, Explore & Clean

In [503]:
import pandas as pd 
import numpy as np  

We import **pandas** for working with tables (DataFrames) and **numpy** for math operations. These are the two most common libraries in any data cleaning project.

In [504]:
titanic = pd.read_csv(
    r"C:\Users\MK\ml-capstone\my-capstone\data\raw\train.csv",
    quotechar='"',
    skipinitialspace=True
)
print(titanic.shape)
print(titanic.head())
print(titanic.info())
print(titanic.describe())

(891, 12)
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

  Name                                                                                  \
0  Braund, Mr. Owen Harris                       ...                                     
1  Cumings, Mrs. John Bradley (Florence Briggs Th...                                     
2  Heikkinen, Miss. Laina                        ...                                     
3  Futrelle, Mrs. Jacques Heath (Lily May Peel)  ...                                     
4  Allen, Mr. William Henry                      ...                                     

   Sex     Age    SibSp  Parch  Ticket              Fare      Cabin            \
0  male     22.0      1      0  A/5 21171             7.2500              NaN   
1  female   38.0      1      0  PC 17599             71.2833  C85     

We load the Titanic CSV with `quotechar` and `skipinitialspace=True` to handle any extra spaces or quoting issues in the file. Then we run 4 commands to understand the data:
- **shape** — gives us 891 rows and 12 columns
- **head()** — shows first 5 rows so we can see what the data looks like
- **info()** — shows data types and how many non-null values per column (this is where we spot missing data: Age has 177 missing, Cabin has 687 missing, Embarked has 2 missing)
- **describe()** — gives stats like mean, min, max for numeric columns (helps spot outliers like max Fare = 512 vs median of 14)

In [505]:
titanic.columns = titanic.columns.str.strip()
titanic["Embarked"].fillna("S", inplace=True) 




Two quick fixes here:
- **str.strip()** removes any extra spaces from column names (can cause KeyError if left in)
- **fillna("S")** fills the 2 missing Embarked values with "S" (Southampton), which is the most common port (72% of passengers)

In [506]:
print(titanic['Age'].dtype)

float64


Checking what type Age is stored as. If it's `object` (text) instead of a number, we can't calculate the median.

In [507]:
titanic['Age'] = pd.to_numeric(titanic['Age'], errors='coerce')

Converts Age to a proper number type. `errors='coerce'` means if any value can't be converted (like a typo), it becomes NaN instead of crashing.

In [508]:
print(titanic['Age'].dtype)
print(titanic['Age'].isnull().sum())

float64
177


Verifying the conversion worked. Age should now be `float64` and we can see how many null values we need to fill.

In [509]:
print(titanic.groupby(["Pclass" , "Sex"])["Age"].median())

Pclass  Sex   
1       female    35.0
        male      40.0
2       female    28.0
        male      30.0
3       female    21.5
        male      25.0
Name: Age, dtype: float64


Looking at the median age for each class + gender group. A 1st class female has a different typical age than a 3rd class male, so filling by group is more accurate than one overall median.

In [510]:
titanic["Age"] = titanic.groupby(["Pclass"  , "Sex"])["Age"].transform(
    lambda x:x.fillna(x.median())
)

The smart fill: for each group (e.g. 1st class males), replaces missing ages with that group's median. `transform()` keeps the same shape as the original column so it fits back in.

In [511]:
titanic["Age"].fillna(titanic["Age"].median(), inplace=True)

Safety net: if any group had ALL ages missing, those rows would still be null after the group fill. This catches any leftovers using the overall median.

In [512]:
print(titanic['Age'].isnull().sum())

0


Final Age check — should print **0**. No more missing ages.

In [513]:
titanic.drop(columns =["Cabin" , "PassengerId" , "Ticket"] , inplace=True) 

Dropping three columns that won't help predict survival:
- **Cabin** — 77% missing, too much to fill
- **PassengerId** — just a row number, no real meaning
- **Ticket** — random codes with no clear pattern

In [514]:
print("Duplicates before:", titanic.duplicated().sum())
titanic.drop_duplicates(inplace=True)
print("Duplicates after:", titanic.duplicated().sum())
print("Shape:", titanic.shape)

Duplicates before: 0
Duplicates after: 0
Shape: (891, 9)


Checking for duplicate rows and removing them. `duplicated()` flags rows that are exact copies of another row. We print the count before and after to confirm they were removed. In Titanic, there are usually 0 duplicates since each passenger is unique, but it's good practice to always check.

In [515]:
print(titanic['Fare'].describe())

count    891.000000
mean      32.204208
std       49.693429
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max      512.329200
Name: Fare, dtype: float64


Looking at Fare stats before handling outliers. The big gap between the 75th percentile and the max tells us there are extreme values.

In [516]:
Q1 = titanic['Fare'].quantile(0.25)
Q3 = titanic['Fare'].quantile(0.75)

IQR = Q3 - Q1
print(IQR)


Lower_limit = Q1 - 1.5 * IQR
Upper_limit = Q3 + 1.5 * IQR

print(Lower_limit, Upper_limit)

23.0896
-26.724 65.6344


Using the **IQR method** to calculate outlier boundaries. IQR = Q3 - Q1 (the middle 50% of data). Anything below Q1 - 1.5×IQR or above Q3 + 1.5×IQR is an outlier. This is the same math behind boxplot whiskers.

In [517]:
titanic['Fare'] = titanic['Fare'].clip(lower=-26.724, upper=65.6344)
print(titanic['Fare'].describe())

count    891.000000
mean      24.046813
std       20.481625
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max       65.634400
Name: Fare, dtype: float64


Instead of deleting outlier rows (which loses data), we **clip** them: any fare above the upper limit gets capped down to it. This keeps all passengers in the dataset while removing extreme values.

In [518]:
print(titanic.dtypes)

Survived      int64
Pclass        int64
Name         object
Sex          object
Age         float64
SibSp         int64
Parch         int64
Fare        float64
Embarked     object
dtype: object


Checking data types before encoding. Sex and Embarked are still `object` (text) — models need numbers, so we convert them next.

In [519]:
# Clean values first
titanic['Sex'] = titanic['Sex'].str.strip()
titanic['Embarked'] = titanic['Embarked'].str.strip()

# Then convert to numbers
titanic['Sex'] = titanic['Sex'].map({"male": 1, "female": 0})
titanic['Embarked'] = titanic['Embarked'].map({"S": 0, "C": 1, "Q": 2})

First we clean any extra spaces from Sex and Embarked using `str.strip()` (prevents mismatches like `"male "` not matching `"male"`). Then we encode both:
- **Sex** — male = 1, female = 0 (binary encoding, only 2 categories)
- **Embarked** — S = 0, C = 1, Q = 2 (label encoding, 3 categories)

In [520]:
titanic.drop(columns =["Name"] , inplace=True) 

Dropping the Name column — it's unique per passenger and can't be used directly by a model.

In [521]:
print(titanic.dtypes)

Survived      int64
Pclass        int64
Sex           int64
Age         float64
SibSp         int64
Parch         int64
Fare        float64
Embarked      int64
dtype: object


Final check: all columns should now be numeric (`int64` or `float64`). No more text columns. The dataset is clean and ready for EDA or modeling.

In [522]:
titanic.to_csv("cleaned_titanic.csv", index=True)

Saving the cleaned dataset to a CSV file so we can use it in the next phases (EDA, modeling) without re-running all the cleaning steps.